# Hands-on Exercise 1 (Modified) — MLP on MNIST, MLflow Comparison
### AI Operations (AIOps) — Module 1, Question 2

Modified from the Lecture 3 Exercise 1 starter notebook:
- Predictor: `RandomForestClassifier` → `MLPClassifier`
- Dataset: `load_iris()` → MNIST (`fetch_openml`)
- Sweep: varies **learning_rate** AND **batch_size** together, 6 runs total
- Metrics are logged **once per epoch** (not just once at the end) so the MLflow UI shows
  a real `train_loss` vs `val_accuracy` trend — needed to answer the overfitting question.

**Deliverable:** a screenshot of the 6-run comparison view, the `run_id` of the best run
(printed at the end via `mlflow.search_runs()`), and the `log_param`/`log_metric` lines below.

> **Prerequisite:** a local MLflow Tracking Server must already be running:
> ```bash
> mlflow server --backend-store-uri sqlite:///mlflow.db \
>     --default-artifact-root ./mlruns --host 0.0.0.0 --port 5000 --allowed-hosts "*" --cors-allowed-origins "http://localhost:5000, http://127.0.0.1:5000"
> ```
> Run that command in a separate terminal *before* executing the cells below, then leave it running.

## Step 0 — Setup
Install dependencies (skip if already installed) and import libraries.

In [ ]:
# !pip install mlflow scikit-learn pandas --quiet

import mlflow
import mlflow.sklearn
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, log_loss

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("mnist-mlp-classifier")
print("Tracking URI:", mlflow.get_tracking_uri())

## Step 1 — Load MNIST instead of IRIS
This replaces the starter notebook's `load_iris()` cell. First run downloads MNIST (~1-2 min); it's cached after that.

In [ ]:
print("Downloading MNIST (this can take a minute or two the first time)...")
mnist = fetch_openml("mnist_784", version=1, as_frame=False)
X = mnist.data / 255.0            # scale pixels 0-1, MLPs train much better this way
y = mnist.target.astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train size: {X_train.shape[0]}   Test size: {X_test.shape[0]}")

## Step 2 — The starter function (un-instrumented)
Same role as the starter notebook's `train_and_evaluate()`, but:
- uses `MLPClassifier` instead of `RandomForestClassifier`
- trains epoch-by-epoch (`warm_start=True`) so we can log a metric **per epoch**
  instead of just one final number at the end

In [ ]:
def train_and_evaluate(learning_rate=0.001, batch_size=32,
                        hidden_layer_sizes=(100,), n_epochs=15):
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layer_sizes,
        learning_rate_init=learning_rate,
        batch_size=batch_size,
        max_iter=1,          # we call .fit() once per epoch ourselves, in the loop below
        warm_start=True,     # keeps training from where it left off each .fit() call
        random_state=42,
    )

    train_loss_history = []
    val_acc_history = []

    for epoch in range(n_epochs):
        model.fit(X_train, y_train)
        train_loss = log_loss(y_train, model.predict_proba(X_train))
        val_acc = accuracy_score(y_test, model.predict(X_test))
        train_loss_history.append(train_loss)
        val_acc_history.append(val_acc)

    return model, train_loss_history, val_acc_history

# Sanity check — no MLflow involved yet, just confirm it trains
_model, _tl, _va = train_and_evaluate(n_epochs=2)
print(f"quick check -> final train_loss={_tl[-1]:.4f}  final val_accuracy={_va[-1]:.4f}")

## Step 3 — Instrument it: manual logging
Wrap training in `with mlflow.start_run():` and log parameters + **per-epoch** metrics + a tag.

This is the exact code needed for **Q2 part 3** (the `mlflow.log_param` / `mlflow.log_metric` lines).

In [ ]:
def train_and_log(learning_rate=0.001, batch_size=32,
                   hidden_layer_sizes=(100,), n_epochs=15, run_name=None):
    with mlflow.start_run(run_name=run_name):
        # --- parameters (at least 3, as in the starter notebook) ---
        mlflow.log_param("model_type", "MLPClassifier")
        mlflow.log_param("learning_rate", learning_rate)
        mlflow.log_param("batch_size", batch_size)
        mlflow.log_param("hidden_layer_sizes", hidden_layer_sizes)
        mlflow.log_param("n_epochs", n_epochs)

        model, train_loss_history, val_acc_history = train_and_evaluate(
            learning_rate, batch_size, hidden_layer_sizes, n_epochs
        )

        # --- metrics, logged ONE PER EPOCH so the UI shows a trend, not a single point ---
        for epoch in range(n_epochs):
            mlflow.log_metric("train_loss", train_loss_history[epoch], step=epoch)
            mlflow.log_metric("val_accuracy", val_acc_history[epoch], step=epoch)

        mlflow.set_tag("team", "data-science")
        mlflow.sklearn.log_model(model, name="model")

        run_id = mlflow.active_run().info.run_id
        final_acc = val_acc_history[-1]
        final_loss = train_loss_history[-1]
        print(f"Logged run {run_id}  |  lr={learning_rate}  bs={batch_size}  "
              f"final_val_acc={final_acc:.4f}  final_train_loss={final_loss:.4f}")
        return run_id

baseline_run_id = train_and_log(learning_rate=0.001, batch_size=32, run_name="mlp-baseline")

## Step 4 — Sweep: 6 runs varying `learning_rate` AND `batch_size`
This replaces the starter notebook's 4-run `n_estimators` sweep. We vary **two** hyperparameters
together (as the assignment requires) across 6 combinations.

Note: `baseline_run_id` above already logged one run (`lr=0.001, bs=32`) — the loop below logs the
remaining 5, so you end up with 6 runs total in the `mnist-mlp-classifier` experiment.

In [ ]:
configs = [
    (0.001, 128),
    (0.001, 512),
    (0.01,  32),
    (0.01,  128),
    (0.01,  512),
]

sweep_run_ids = [baseline_run_id]
for lr, bs in configs:
    rid = train_and_log(
        learning_rate=lr, batch_size=bs,
        run_name=f"mlp-lr{lr}-bs{bs}",
    )
    sweep_run_ids.append(rid)

print("\nAll 6 run IDs:", sweep_run_ids)

## Step 5 — Find the best run with `mlflow.search_runs()`
No need to open the UI to find the winner — query it directly, same idea as the starter notebook's Step 6.

In [ ]:
runs_df = mlflow.search_runs(
    experiment_names=["mnist-mlp-classifier"],
    order_by=["metrics.val_accuracy DESC"],
)

display_cols = [c for c in runs_df.columns if c in (
    "run_id", "tags.mlflow.runName", "params.learning_rate",
    "params.batch_size", "metrics.val_accuracy", "metrics.train_loss",
)]
print(runs_df[display_cols].to_string(index=False))

best_run = runs_df.iloc[0]
print(f"\nBest run: {best_run['run_id']}  (val_accuracy={best_run['metrics.val_accuracy']:.4f})")

## Step 6 — Open the MLflow UI
1. Go to **http://localhost:5000** in your browser.
2. Open the **mnist-mlp-classifier** experiment.
3. Select all 6 runs from this notebook (checkboxes on the left) and click **Compare**.
4. Click into any single run and check its `train_loss` / `val_accuracy` charts over the 15 epochs
   — this is what you need to describe the overfitting trend in your written analysis.
5. Confirm the run the UI ranks highest matches the `best_run` printed above.

---
### ✅ Deliverable checklist
- [ ] Screenshot of the 6-run comparison view in the MLflow UI
- [ ] The `run_id` of the best run (printed above by `mlflow.search_runs()`)
- [ ] 150–250 word analysis: best run + why, overfitting evidence from the train_loss/val_accuracy trend, which hyperparameter (learning_rate or batch_size) had the larger effect
- [ ] The `mlflow.log_param` / `mlflow.log_metric` lines from the `train_and_log()` cell above